# 12 — Tabular Neural Network Growth Prediction Prototype

This notebook documents Step 36: a real feed-forward neural-network prototype for structured dog growth data.

The model predicts a conservative review signal: `normal_growth` vs `needs_attention`. It is not a veterinary diagnosis, not an official Cane Corso certification, and not a photo-based breed-purity proof.


## Why tabular first?

An image neural network requires a licensed, labeled Cane Corso photo dataset. Until that dataset exists, the honest neural-network step is to train on structured growth data.

This notebook uses `scikit-learn`'s `MLPClassifier`, which is a real multi-layer perceptron neural network.


In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [ ]:
dataset_path = Path('../data/processed/dog_growth_classification_sample.csv')
df = pd.read_csv(dataset_path)
df.head()


## Features and target

The prototype uses age, weight, average adult breed weight and gender. It deliberately avoids using `bcs_predicted` as an input because that field can be too close to the review target and can create leakage-like behavior.


In [ ]:
numeric_features = ['visit_age_months', 'weight_kg', 'average_adult_breed_weight_kg']
categorical_features = ['gender']
target = 'growth_status_binary'

X = df[numeric_features + categorical_features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=36
)


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)

model = MLPClassifier(
    hidden_layer_sizes=(16, 8),
    activation='relu',
    solver='adam',
    alpha=0.0005,
    learning_rate_init=0.005,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=36,
)

pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('neural_network', model),
])

pipeline.fit(X_train, y_train)


In [ ]:
predictions = pipeline.predict(X_test)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, predictions, average='binary', zero_division=0
)

{
    'accuracy': round(float(accuracy_score(y_test, predictions)), 4),
    'precision_needs_attention': round(float(precision), 4),
    'recall_needs_attention': round(float(recall), 4),
    'f1_needs_attention': round(float(f1), 4),
    'confusion_matrix': confusion_matrix(y_test, predictions, labels=[0, 1]).tolist(),
}


## Safety boundary

This is a real neural-network prototype, but it remains educational. It is not a veterinary diagnostic system, not a breeding decision system, not an official registry decision and not a photo-based breed-purity model.

The image neural network remains future work until a licensed labeled image dataset exists.
